# End-to-End Metrics
### Semantic Similarity, Aspect Critic, Noise Sensitivity — RAGAS

Corpus: `OWASP Top 10 for LLM Applications (2025)`. Same 8 ground-truth questions as the other two RAGAS notebooks this week. These three metrics look at the pipeline as a whole rather than isolating retrieval or generation: how close the final answer is to the reference in meaning, whether it passes a rule we define ourselves, and whether the system stays correct when the retrieved context is contaminated with something irrelevant.

## Step 1: Build the RAG pipeline

In [1]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf ragas openai python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

PDF_PATH = "OWASP-Top-10-for-LLMs-v2025.pdf"

pages = PyPDFLoader(PDF_PATH).load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vector_store = FAISS.from_documents(chunks, embeddings)

rag_llm = ChatOllama(model="llama3.2:3b", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_5612\159563066.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
incorrect startxref pointer(1)
parsing for Object Streams
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set
Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: Ground truth — questions and reference answers
Same 8 questions used across all three RAGAS notebooks this week.

In [2]:
ground_truth = [
    {
        "question": "What is Prompt Injection, according to LLM01?",
        "reference": "A Prompt Injection Vulnerability occurs when user prompts alter the LLM's behavior or output in unintended ways. The input does not need to be human-readable, only something the model parses.",
    },
    {
        "question": "What kinds of sensitive information can an LLM application expose, according to LLM02?",
        "reference": "Personal identifiable information (PII), financial details, health records, confidential business data, security credentials, legal documents, and proprietary training methods or source code.",
    },
    {
        "question": "What real attack against a model hosted on Hugging Face is cited as an example of Supply Chain risk under LLM03?",
        "reference": "PoisonGPT: an attacker bypassed Hugging Face's safety features by directly tampering with a model's parameters to spread misinformation.",
    },
    {
        "question": "What is Excessive Agency, according to LLM06?",
        "reference": "Excessive Agency is the vulnerability where an LLM-based system is granted excessive functionality, permissions, or autonomy to call functions or tools, letting it take unintended or damaging actions.",
    },
    {
        "question": "What does System Prompt Leakage warn about, according to LLM07?",
        "reference": "System prompts can inadvertently contain sensitive information, such as credentials or internal business rules, that was not meant to be discovered; the system prompt should never be treated as a secret or used as a security control.",
    },
    {
        "question": "What security risks affect vectors and embeddings in RAG systems, according to LLM08?",
        "reference": "Weaknesses in how vectors and embeddings are generated, stored, or retrieved can be exploited to inject harmful content, manipulate outputs, or leak sensitive data -- for example, in a multi-tenant vector database, one group's embeddings could be retrieved in response to another group's queries.",
    },
    {
        "question": "What is the main cause of misinformation in LLMs, according to LLM09?",
        "reference": "Hallucination -- the LLM generates content that seems accurate but is fabricated, filling gaps in its training data using statistical patterns without truly understanding the content.",
    },
    {
        "question": "What is Unbounded Consumption, according to LLM10?",
        "reference": "A risk where an LLM application allows excessive, uncontrolled inference operations, which can lead to denial of service, runaway costs, or model theft through extraction and cloning attacks.",
    },
]

import pandas as pd
pd.DataFrame(ground_truth)

,question,reference
0,"What is Prompt Injection, according to LLM01?",A Prompt Injection Vulnerability occurs when u...
1,What kinds of sensitive information can an LLM...,"Personal identifiable information (PII), finan..."
2,What real attack against a model hosted on Hug...,PoisonGPT: an attacker bypassed Hugging Face's...
3,"What is Excessive Agency, according to LLM06?",Excessive Agency is the vulnerability where an...
4,"What does System Prompt Leakage warn about, ac...",System prompts can inadvertently contain sensi...
5,What security risks affect vectors and embeddi...,Weaknesses in how vectors and embeddings are g...
6,What is the main cause of misinformation in LL...,Hallucination -- the LLM generates content tha...
7,"What is Unbounded Consumption, according to LL...",A risk where an LLM application allows excessi...


## Step 3: Retrieve + generate the "clean" answer for each question

In [3]:
RAG_PROMPT = """Answer the question using only the following context. Be concise.

Context:
{context}

Question: {question}
Answer:"""

for item in ground_truth:
    docs = vector_store.similarity_search(item["question"], k=3)
    item["contexts"] = [doc.page_content for doc in docs]
    context = "\n\n".join(item["contexts"])
    item["answer"] = rag_llm.invoke(
        RAG_PROMPT.format(context=context, question=item["question"])
    ).content.strip()

## Step 4: Build a "noisy" version of the context and regenerate
For each question, one chunk retrieved for a *different, unrelated* question is mixed into the context — a topic-shifted distractor, the kind a bigger `top-k` or a looser similarity threshold can let through in a real system.

In [4]:
for i, item in enumerate(ground_truth):
    distractor_item = ground_truth[(i + 3) % len(ground_truth)]
    distractor_chunk = vector_store.similarity_search(
        distractor_item["question"], k=1
    )[0].page_content
    item["noisy_contexts"] = item["contexts"] + [distractor_chunk]

    noisy_context_text = "\n\n".join(item["noisy_contexts"])
    item["noisy_answer"] = rag_llm.invoke(
        RAG_PROMPT.format(context=noisy_context_text, question=item["question"])
    ).content.strip()

for item in ground_truth[:2]:
    print("Q:", item["question"])
    print("Clean answer:", item["answer"])
    print("Noisy answer:", item["noisy_answer"])
    print()

Q: What is Prompt Injection, according to LLM01?
Clean answer: Prompt Injection refers to the vulnerability in Large Language Models (LLMs) where an attacker can inject malicious or biased prompts to manipulate the model's output.
Noisy answer: According to LLM01:2025 Prompt Injection (Table of Contents, page 3), Prompt Injection refers to a type of vulnerability where an attacker injects malicious prompts into a large language model (LLM) to manipulate its output and potentially extract sensitive information.

Q: What kinds of sensitive information can an LLM application expose, according to LLM02?
Clean answer: According to LLM02, an LLM application can expose:

1. Personal identifiable information (PII)
2. Proprietary algorithms or data
3. Confidential business data
4. Security credentials
5. Legal documents
6. Sensitive system architecture
7. API keys
8. Database credentials
9. User tokens
Noisy answer: According to LLM02, an LLM application can expose:

1. Personal identifiable in

## Step 5: Wire RAGAS to an LLM judge
Same reasoning as the other two notebooks: judging needs a reliable instruction-follower, so `gpt-4o-mini` is the judge. Retrieval and generation stay fully local on Ollama.

In [5]:
import sys, types

_stub = types.ModuleType("langchain_community.chat_models.vertexai")


class ChatVertexAI:
    pass


_stub.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _stub

from dotenv import load_dotenv
load_dotenv()

from openai import AsyncOpenAI
from ragas.llms.base import llm_factory
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings

client = AsyncOpenAI()
judge_llm = llm_factory("gpt-4o-mini", client=client)
judge_embeddings = RagasOpenAIEmbeddings(client=client, model="text-embedding-3-small")

print("Judge ready.")

Judge ready.


## Step 6: Semantic Similarity — meaning match to the reference
Embeds the clean answer and the reference, then scores their cosine similarity — the same computation as Answer Similarity in the Generation Metrics notebook, viewed here as an end-to-end sanity check rather than a generation-only one.

In [6]:
from ragas.metrics.collections import SemanticSimilarity
semantic_similarity_metric = SemanticSimilarity(embeddings=judge_embeddings)

for item in ground_truth:
    result = await semantic_similarity_metric.ascore(
        reference=item["reference"],
        response=item["answer"],
    )
    item["semantic_similarity"] = result.value

pd.DataFrame(ground_truth)[["question", "semantic_similarity"]]

,question,semantic_similarity
0,"What is Prompt Injection, according to LLM01?",0.832064
1,What kinds of sensitive information can an LLM...,0.662357
2,What real attack against a model hosted on Hug...,0.651935
3,"What is Excessive Agency, according to LLM06?",0.764107
4,"What does System Prompt Leakage warn about, ac...",0.573637
5,What security risks affect vectors and embeddi...,0.678906
6,What is the main cause of misinformation in LL...,0.685274
7,"What is Unbounded Consumption, according to LL...",0.456040


## Step 7: Aspect Critic — a custom rubric, not a generic one
RAGAS sends the rubric and the answer to the judge LLM, which returns a binary pass/fail. Here the rubric is domain-specific to this course rather than a generic "is this a good answer" check: does the answer stay inside OWASP terminology, or does it invent security concepts the document never mentions?

In [7]:
from ragas.metrics.collections import DomainSpecificRubrics

rubrics = {
    "score1_description": (
        "The answer is grounded in OWASP Top 10 for LLM Applications terminology and does not "
        "introduce security concepts, attacks, or mitigations absent from that document."
    ),
    "score0_description": (
        "The answer introduces security concepts, attacks, or mitigations not found in the "
        "OWASP Top 10 for LLM Applications document, or contradicts it."
    ),
}
aspect_critic_metric = DomainSpecificRubrics(llm=judge_llm, rubrics=rubrics)

for item in ground_truth:
    result = await aspect_critic_metric.ascore(
        user_input=item["question"],
        response=item["answer"],
    )
    item["aspect_critic"] = result.value
    item["aspect_critic_reason"] = result.reason

pd.DataFrame(ground_truth)[["question", "aspect_critic", "aspect_critic_reason"]]

,question,aspect_critic,aspect_critic_reason
0,"What is Prompt Injection, according to LLM01?",0.0,The response introduces the concept of Prompt ...
1,What kinds of sensitive information can an LLM...,1.0,The response lists various types of sensitive ...
2,What real attack against a model hosted on Hug...,2.0,The response identifies 'PoisonGPT' as an exam...
3,"What is Excessive Agency, according to LLM06?",0.0,The response does not provide a definition or ...
4,"What does System Prompt Leakage warn about, ac...",1.0,The response accurately describes the concept ...
5,What security risks affect vectors and embeddi...,0.0,The response introduces security concepts and ...
6,What is the main cause of misinformation in LL...,0.0,The response introduces concepts such as hallu...
7,"What is Unbounded Consumption, according to LL...",0.0,The response introduces the concept of 'Unboun...


## Step 8: Noise Sensitivity — does the distractor cause a wrong claim?
Decomposes the answer into atomic statements, checks each one's faithfulness against the reference and against every retrieved chunk, then reports the fraction of statements that are *both* wrong (unsupported by the reference) *and* traceable specifically to one context group. `mode="relevant"` scores the clean run; `mode="irrelevant"` scores the noisy run, isolating errors caused by the injected distractor specifically. This is by far the most LLM-call-heavy metric in this notebook (it re-checks every statement against every retrieved chunk), so it runs on a 4-question sample instead of all 8 to keep the notebook's runtime reasonable.

In [8]:
from ragas.metrics.collections import NoiseSensitivity

noise_relevant_metric = NoiseSensitivity(llm=judge_llm, mode="relevant")
noise_irrelevant_metric = NoiseSensitivity(llm=judge_llm, mode="irrelevant")

noise_sample = ground_truth[:4]

for item in noise_sample:
    relevant_result = await noise_relevant_metric.ascore(
        user_input=item["question"],
        response=item["answer"],
        reference=item["reference"],
        retrieved_contexts=item["contexts"],
    )
    irrelevant_result = await noise_irrelevant_metric.ascore(
        user_input=item["question"],
        response=item["noisy_answer"],
        reference=item["reference"],
        retrieved_contexts=item["noisy_contexts"],
    )
    item["noise_sensitivity_clean"] = relevant_result.value
    item["noise_sensitivity_noisy"] = irrelevant_result.value

pd.DataFrame(noise_sample)[["question", "noise_sensitivity_clean", "noise_sensitivity_noisy"]]

,question,noise_sensitivity_clean,noise_sensitivity_noisy
0,"What is Prompt Injection, according to LLM01?",0.000000,0.0
1,What kinds of sensitive information can an LLM...,0.555556,0.0
2,What real attack against a model hosted on Hug...,0.000000,0.0
3,"What is Excessive Agency, according to LLM06?",0.000000,0.5


A non-zero `noise_sensitivity_noisy` value pinpoints a claim the model only made because of the injected distractor chunk — that row is worth reading by hand. A `0.0` across the board is also a real result: it means this pipeline's answers stayed grounded even with a topic-shifted distractor mixed into the context, for this sample.

## Step 9: Scorecard

In [9]:
generation_score_cols = ["semantic_similarity", "aspect_critic"]
generation_scorecard = pd.DataFrame(ground_truth)[["question"] + generation_score_cols]
generation_scorecard.loc["mean", "question"] = ""
generation_scorecard.loc["mean", generation_score_cols] = generation_scorecard[generation_score_cols].mean()
generation_scorecard

,question,semantic_similarity,aspect_critic
0,"What is Prompt Injection, according to LLM01?",0.832064,0.0
1,What kinds of sensitive information can an LLM...,0.662357,1.0
2,What real attack against a model hosted on Hug...,0.651935,2.0
3,"What is Excessive Agency, according to LLM06?",0.764107,0.0
4,"What does System Prompt Leakage warn about, ac...",0.573637,1.0
5,What security risks affect vectors and embeddi...,0.678906,0.0
6,What is the main cause of misinformation in LL...,0.685274,0.0
7,"What is Unbounded Consumption, according to LL...",0.456040,0.0
mean,,0.663040,0.5


In [10]:
noise_score_cols = ["noise_sensitivity_clean", "noise_sensitivity_noisy"]
noise_scorecard = pd.DataFrame(noise_sample)[["question"] + noise_score_cols]
noise_scorecard.loc["mean", "question"] = ""
noise_scorecard.loc["mean", noise_score_cols] = noise_scorecard[noise_score_cols].mean()
noise_scorecard

,question,noise_sensitivity_clean,noise_sensitivity_noisy
0,"What is Prompt Injection, according to LLM01?",0.000000,0.000
1,What kinds of sensitive information can an LLM...,0.555556,0.000
2,What real attack against a model hosted on Hug...,0.000000,0.000
3,"What is Excessive Agency, according to LLM06?",0.000000,0.500
mean,,0.138889,0.125


In [11]:
# Production targets from the slide deck's scorecard.
# Noise Sensitivity is 'lower is better', so it's checked with <= instead of >=.
gen_means = generation_scorecard.loc["mean", generation_score_cols]
noise_means = noise_scorecard.loc["mean", noise_score_cols]

print(f"semantic_similarity      mean={gen_means['semantic_similarity']:.2f}  minimum=0.75  -> {'PASS' if gen_means['semantic_similarity'] >= 0.75 else 'BELOW MINIMUM'}")
print(f"aspect_critic            mean={gen_means['aspect_critic']:.2f}  (pass rate on the custom rubric -- no fixed target, judged case by case)")
print(f"noise_sensitivity_clean  mean={noise_means['noise_sensitivity_clean']:.2f}  ideal<=0.10  -> {'PASS' if noise_means['noise_sensitivity_clean'] <= 0.10 else 'INVESTIGATE'}")
print(f"noise_sensitivity_noisy  mean={noise_means['noise_sensitivity_noisy']:.2f}  acceptable<=0.20  -> {'PASS' if noise_means['noise_sensitivity_noisy'] <= 0.20 else 'INVESTIGATE'}")

semantic_similarity      mean=0.66  minimum=0.75  -> BELOW MINIMUM
aspect_critic            mean=0.50  (pass rate on the custom rubric -- no fixed target, judged case by case)
noise_sensitivity_clean  mean=0.14  ideal<=0.10  -> INVESTIGATE
noise_sensitivity_noisy  mean=0.12  acceptable<=0.20  -> PASS


## Try it yourself
1. Look at the row where `noise_sensitivity_noisy` is highest and read that question's `noisy_answer` by hand — confirm the distractor chunk really did cause the extra (wrong) claim.
2. Rewrite the Aspect Critic rubric to check something else entirely — e.g. "does the answer mention which LLM0X code the risk belongs to" — and see which questions start failing it.
3. Compare `semantic_similarity` here against `answer_similarity` in the Generation Metrics notebook for the same questions — they're the same underlying computation, so they should match almost exactly.